In [3]:
# Дополнительные импорты
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, mean_absolute_error, classification_report, confusion_matrix
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

In [8]:
# Загрузка и подготовка данных (используем ваш код)
df = pd.read_csv('D:/my_ML/diploma_polytech/data/raw/vehicle_ins_data_1.csv', sep = ";",index_col= False)
df['Date_start_contract'] = pd.to_datetime(df['Date_start_contract'], errors='coerce')
df['claim_event'] = (df['N_claims_year'] > 1).astype(int)

# Исключаем признаки, приводящие к утечке информации
leakage_features = ['Cost_claims_year', 'N_claims_history']
drop_cols = leakage_features + ['ID_policy','Date_start_contract','End_date','Date_birth',
                                'Renew_date','N_claims_year']

# Отбираем фичи
features = [c for c in df.columns if c not in drop_cols + ['claim_event']]
df = df[features + ['claim_event']].dropna()



C:\Users\andre\AppData\Local\Temp\ipykernel_20128\1361882663.py:2: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('D:/my_ML/diploma_polytech/data/raw/vehicle_ins_data_1.csv', sep = ";",index_col= False)


In [9]:
# Кодируем категориальные переменные
categorical = ['Distribution_channel','Type_risk','Type_fuel']
df = pd.get_dummies(df, columns=[c for c in categorical if c in df.columns], drop_first=True)

# Сплит по времени: train = до 2020, test = 2020+
train_df = df[df['Date_start_contract'] < '2017-07-01'].copy()
test_df  = df[df['Date_start_contract'] >= '2017-07-01'].copy()

# Убираем Start_date после разделения
train_df = train_df.drop(columns=['Date_start_contract'], errors='ignore')
test_df = test_df.drop(columns=['Date_start_contract'], errors='ignore')

# Разделяем X и y
X_train, y_train = train_df.drop('claim_event', axis=1), train_df['claim_event']
X_test, y_test   = test_df.drop('claim_event', axis=1), test_df['claim_event']

# Масштабирование (важно для логистической регрессии)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)



KeyError: 'Date_start_contract'

In [ ]:
# Базовая логистическая регрессия
logreg = LogisticRegression(
    random_state=42,
    max_iter=1000,
    class_weight='balanced'  # для обработки дисбаланса классов
)



In [ ]:
# Обучение модели
logreg.fit(X_train_scaled, y_train)



In [ ]:
# Предсказание вероятностей
y_pred_proba = logreg.predict_proba(X_test_scaled)[:, 1]

# Оценка модели
auc_logreg = roc_auc_score(y_test, y_pred_proba)

print("=" * 50)
print("ЛОГИСТИЧЕСКАЯ РЕГРЕССИЯ - РЕЗУЛЬТАТЫ:")
print(f"ROC-AUC: {auc_logreg:.4f}")
print("=" * 50)

# Детальная диагностика
y_pred = logreg.predict(X_test_scaled)
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# Сравнение с вашей нейросетевой моделью
print("\n" + "=" * 50)
print "СРАВНЕНИЕ МОДЕЛЕЙ:")
print(f"Нейросеть ROC-AUC: 0.96")
print(f"Логистическая регрессия ROC-AUC: {auc_logreg:.4f}")
print("=" * 50)

# Анализ важности признаков
if abs(auc_logreg - 0.96) < 0.05:
    print("\nВНИМАНИЕ: Обе модели показывают высокий ROC-AUC")
    print("Возможные причины:")
    print("1. Сильный дисбаланс классов в данных")
    print("2. Признаки слишком хорошо разделяют классы")
    print("3. Возможна утечка данных (проверьте исключенные признаки)")
    
    # Проверяем баланс классов
    print(f"\nРаспределение классов в train: {np.bincount(y_train)}")
    print(f"Распределение классов в test: {np.bincount(y_test)}")
else:
    print(f"\nЛогистическая регрессия показывает ROC-AUC {auc_logreg:.4f}")
    print("Это suggests что нейросеть может быть переобучена")
    print("Рекомендуется усилить регуляризацию в нейросети")